In [2]:
%%bash
pip install -q streamlit reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.4 MB/s eta 0:00:00


In [3]:
import streamlit as st
import pandas as pd
import numpy as np
import io
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

# ==========================================
# 1. PAGE CONFIGURATION & STYLING
# ==========================================
st.set_page_config(
    page_title="CII RT-153 QC Staffing & Rework Predictor",
    page_icon="🏗️",
    layout="wide"
)

# Custom CSS for UI styling
st.markdown("""
<style>
    .main-header {
        font-size: 2.2rem;
        font-weight: 700;
        color: #1E3A8A;
        margin-bottom: 0.2rem;
    }
    .sub-header {
        font-size: 1.0rem;
        color: #4B5563;
        margin-bottom: 1.5rem;
    }
    .metric-card {
        background-color: #F3F4F6;
        border-radius: 8px;
        padding: 15px;
        border-left: 5px solid #1E3A8A;
    }
</style>
""", unsafe_allow_html=True)

# ==========================================
# 2. MODEL PARAMETERS & CONSTANTS
# ==========================================
# Facility Risk Multipliers (M_type) based on Appendix D process intensity
FACILITY_MULTIPLIERS = {
    "Chemical Plant": 1.25,
    "Oil/Gas Production Facility": 1.25,
    "Petrochemical Facility": 1.25,
    "Power Plant": 1.00,
    "Paper Plant": 1.00,
    "Food Processing Plant": 1.00,
    "Manufacturing Facility": 1.00,
    "Steel/Aluminum Mill": 1.00,
    "Pharmaceutical Plant": 1.10,
    "Water/Wastewater Treatment": 0.85,
    "Transportation Facility": 0.80,
    "General Building": 0.75
}

# Design Readiness Multipliers (M_engr) based on Appendix D completion groups
ENGINEERING_COMPLETION_MULTIPLIERS = {
    "< 20% Complete": 1.50,
    "21% - 40% Complete": 1.35,
    "41% - 60% Complete": 1.20,
    "61% - 80% Complete": 1.00,
    "81% - 95% Complete": 0.85,
    "> 95% Complete": 0.80
}

# Craft Discipline Risk Multipliers (w_d) derived from historical rework intensity
CRAFT_RISK_WEIGHTS = {
    "Piping": 1.83,
    "Electrical & Instrumentation": 1.10,
    "Equipment Installation": 0.90,
    "Civil": 0.70,
    "Structural": 0.70,
    "Other Crafts": 0.60
}

# ==========================================
# 3. HELPER FUNCTIONS & CALCULATIONS
# ==========================================
def calculate_analytics(cost_m, facility_type, eng_comp, craft_shares):
    """Performs Project Exposure Index, Rework %, and QC Staffing calculations."""
    m_type = FACILITY_MULTIPLIERS.get(facility_type, 1.00)
    m_engr = ENGINEERING_COMPLETION_MULTIPLIERS.get(eng_comp, 1.00)

    # Project Exposure Index (PEI)
    pei = cost_m * m_type * m_engr

    # Base QC Staffing Headcount (1 Inspector per $10M PEI, min 1.0)
    base_qc = max(1.0, round(pei / 10.0, 1))

    # Calculate Weighted Discipline Shares
    weighted_shares = {}
    total_weighted_share = 0.0
    for craft, pct in craft_shares.items():
        w = CRAFT_RISK_WEIGHTS.get(craft, 0.70)
        w_val = (pct / 100.0) * w
        weighted_shares[craft] = w_val
        total_weighted_share += w_val

    # Discipline Staffing Breakdown (FTEs)
    qc_breakdown = {}
    for craft, w_val in weighted_shares.items():
        if total_weighted_share > 0:
            qc_breakdown[craft] = round(base_qc * (w_val / total_weighted_share), 2)
        else:
            qc_breakdown[craft] = 0.0

    # Predicted Rework Percentage Calculation
    piping_pct = craft_shares.get("Piping", 0.0)
    predicted_rework = 1.5 + (m_engr * m_type * 2.8) + ((piping_pct / 100.0) * 2.5)
    predicted_rework = min(28.0, max(0.5, round(predicted_rework, 2)))

    # Risk Assessment Category
    if pei < 30:
        risk_level = "Low Risk"
    elif pei <= 75:
        risk_level = "Moderate Risk"
    else:
        risk_level = "High Risk"

    return {
        "pei": round(pei, 2),
        "base_qc": base_qc,
        "predicted_rework": predicted_rework,
        "risk_level": risk_level,
        "qc_breakdown": qc_breakdown,
        "m_type": m_type,
        "m_engr": m_engr
    }

def generate_pdf_report(project_name, cost_m, facility_type, delivery_method, contract_type, eng_comp, craft_shares, results):
    """Generates a PDF summary report using ReportLab."""
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
    story = []

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        'DocTitle',
        parent=styles['Heading1'],
        fontSize=20,
        textColor=colors.HexColor("#1E3A8A"),
        spaceAfter=12
    )
    h2_style = ParagraphStyle(
        'Heading2',
        parent=styles['Heading2'],
        fontSize=13,
        textColor=colors.HexColor("#1F2937"),
        spaceBefore=10,
        spaceAfter=6
    )
    body_style = styles['Normal']

    # Header
    story.append(Paragraph("CII RT-153 Quality Control Staffing & Rework Analysis", title_style))
    story.append(Paragraph(f"<b>Project Identification:</b> {project_name}", body_style))
    story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#1E3A8A"), spaceAfter=15))

    # Executive Summary Metrics
    story.append(Paragraph("Executive Summary & Predictive Results", h2_style))
    summary_data = [
        ["Estimated Project Cost", f"${cost_m:,.2f} Million", "Project Exposure Index (PEI)", f"{results['pei']}"],
        ["Predicted Field Rework %", f"{results['predicted_rework']}%", "Overall Risk Assessment", results['risk_level']],
        ["Total Recommended QC Staff", f"{results['base_qc']} Inspectors", "Eng. Completion at Start", eng_comp]
    ]
    t_summary = Table(summary_data, colWidths=[140, 120, 150, 110])
    t_summary.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor("#F3F4F6")),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#D1D5DB")),
        ('FONTNAME', (0,0), (-1,-1), 'Helvetica-Bold'),
        ('FONTNAME', (1,0), (1,-1), 'Helvetica'),
        ('FONTNAME', (3,0), (3,-1), 'Helvetica'),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('PADDING', (0,0), (-1,-1), 6),
    ]))
    story.append(t_summary)
    story.append(Spacer(1, 12))

    # Project Parameters
    story.append(Paragraph("Project Configuration Parameters", h2_style))
    config_data = [
        ["Facility Sector", facility_type, "Delivery Method", delivery_method],
        ["Contract Type", contract_type, "Facility Multiplier (M_type)", str(results['m_type'])],
        ["Engineering Status", eng_comp, "Design Readiness Multiplier", str(results['m_engr'])]
    ]
    t_config = Table(config_data, colWidths=[140, 120, 150, 110])
    t_config.setStyle(TableStyle([
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#E5E7EB")),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('PADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(t_config)
    story.append(Spacer(1, 12))

    # Staffing Allocation Table
    story.append(Paragraph("Recommended Inspector Staffing Allocation by Discipline", h2_style))
    staff_data = [["Craft Discipline", "Workhour Share (%)", "Risk Multiplier", "Recommended QC Staff (FTE)"]]
    for craft, fte in results['qc_breakdown'].items():
        share = craft_shares.get(craft, 0.0)
        weight = CRAFT_RISK_WEIGHTS.get(craft, 0.70)
        staff_data.append([craft, f"{share:.1f}%", f"{weight:.2f}", f"{fte:.2f} Inspectors"])

    t_staff = Table(staff_data, colWidths=[160, 110, 110, 140])
    t_staff.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#1E3A8A")),
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#D1D5DB")),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('ALIGN', (1,0), (-1,-1), 'CENTER'),
        ('PADDING', (0,0), (-1,-1), 6),
    ]))
    story.append(t_staff)
    story.append(Spacer(1, 15))

    # Methodology Footnote
    story.append(Paragraph("<i>Note: Predictive calculations are grounded in Construction Industry Institute (CII) Research Topic 153 empirical data (Appendix D). Risk weightings prioritize high-rework disciplines (e.g., Piping) to optimize field quality oversight.</i>", ParagraphStyle('Footnote', parent=styles['Italic'], fontSize=8, textColor=colors.HexColor("#6B7280"))))

    doc.build(story)
    buffer.seek(0)
    return buffer

# ==========================================
# 4. STREAMLIT USER INTERFACE
# ==========================================
st.markdown('<div class="main-header">CII RT-153 Quality Control Staffing & Rework Analytics</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">Predictive Field Rework Modeling & QC Inspector Staffing Allocation App</div>', unsafe_allow_html=True)

# Layout Columns: Input Controls (Left), Visuals & Output (Right)
col_input, col_display = st.columns([1, 1.2], gap="large")

with col_input:
    st.subheader("📋 Project Configuration")

    project_name = st.text_input("Project Name / Identifier", value="Industrial Facility Project Alpha")

    col_c1, col_c2 = st.columns(2)
    with col_c1:
        cost_m = st.number_input("Estimated Cost ($ Millions)", min_value=0.1, max_value=2000.0, value=50.0, step=5.0)
        facility_type = st.selectbox("Facility Sector", list(FACILITY_MULTIPLIERS.keys()), index=0)
        delivery_method = st.selectbox("Project Delivery Method", ["EPC (Engineer-Procure-Construct)", "Design-Bid-Build", "Construction Management", "Other"], index=0)

    with col_c2:
        eng_comp = st.selectbox("Engineering Complete at Constr. Start", list(ENGINEERING_COMPLETION_MULTIPLIERS.keys()), index=2)
        contract_type = st.selectbox("Contract Type", ["Cost-Reimbursable / Cost Plus", "Cost Plus with Incentives", "Fixed Fee / Lump Sum", "Other"], index=0)

    st.markdown("---")
    st.subheader("👷 Craft Workhour Density (%)")
    st.caption("Adjust percentages to reflect expected direct craft labor distribution (auto-normalizes to 100%).")

    piping = st.slider("Piping Craft %", 0, 100, 35)
    elec = st.slider("Electrical & Instrumentation %", 0, 100, 20)
    civil = st.slider("Civil Craft %", 0, 100, 15)
    struct = st.slider("Structural Craft %", 0, 100, 10)
    eqpt = st.slider("Equipment Installation %", 0, 100, 10)
    other = st.slider("Other Crafts %", 0, 100, 10)

    # Calculate normalized craft shares
    raw_shares = {
        "Piping": piping,
        "Electrical & Instrumentation": elec,
        "Civil": civil,
        "Structural": struct,
        "Equipment Installation": eqpt,
        "Other Crafts": other
    }
    raw_total = sum(raw_shares.values())

    if raw_total > 0:
        craft_shares = {k: (v / raw_total) * 100.0 for k, v in raw_shares.items()}
    else:
        craft_shares = {k: 16.67 for k in raw_shares.keys()}

    if raw_total != 100:
        st.warning(f"Raw inputs sum to {raw_total}%. Automatically normalized to 100% for calculation.")

# ==========================================
# 5. ANALYTICS COMPUTATION & DISPLAY
# ==========================================
results = calculate_analytics(cost_m, facility_type, eng_comp, craft_shares)

with col_display:
    st.subheader("📊 Predictive Metrics & Staffing Plan")

    # KPI Summary Cards
    m1, m2, m3 = st.columns(3)
    with m1:
        st.metric("Predicted Field Rework", f"{results['predicted_rework']}%")
    with m2:
        st.metric("Total QC Inspectors", f"{results['base_qc']} FTE")
    with m3:
        st.metric("Project Exposure Index", f"{results['pei']}")

    st.markdown("---")

    # Staffing Allocation Data Table & Bar Chart
    st.markdown("### Recommended Inspector Allocation by Discipline")

    df_staffing = pd.DataFrame([
        {
            "Craft Discipline": craft,
            "Workhour Share (%)": f"{craft_shares[craft]:.1f}%",
            "Risk Weight": CRAFT_RISK_WEIGHTS[craft],
            "Inspectors (FTE)": fte
        }
        for craft, fte in results['qc_breakdown'].items()
    ])

    st.dataframe(df_staffing, use_container_width=True, hide_index=True)

    # Visualization Chart
    chart_data = pd.DataFrame({
        "Discipline": list(results['qc_breakdown'].keys()),
        "Inspectors (FTE)": list(results['qc_breakdown'].values())
    }).set_index("Discipline")

    st.bar_chart(chart_data)

    # Report Generation & Download
    st.markdown("---")
    st.markdown("### 📄 Export Official Assessment")

    pdf_buffer = generate_pdf_report(
        project_name=project_name,
        cost_m=cost_m,
        facility_type=facility_type,
        delivery_method=delivery_method,
        contract_type=contract_type,
        eng_comp=eng_comp,
        craft_shares=craft_shares,
        results=results
    )

    st.download_button(
        label="📥 Download PDF Staffing Report",
        data=pdf_buffer,
        file_name=f"QC_Staffing_Report_{project_name.replace(' ', '_')}.pdf",
        mime="application/pdf",
        use_container_width=True
    )

2026-10-05 19:11:42.553 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 19:11:42.558 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 19:11:42.977 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-05 19:11:42.982 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 19:11:42.983 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 19:11:42.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 19:11:42.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn